# Tutorial: Interconnection Financing Triage

## What is this notebook?

This notebook is a step-by-step tutorial for analyzing EU cross-border electricity interconnection projects and classifying them into **financing tracks**. The goal is to estimate how much of the ~170 bn EUR TYNDP investment pipeline can be market-financed vs. how much needs targeted EU support.

### The three financing tracks

| Track | Description | Financing mix |
|---|---|---|
| **Track 1** | Commercially viable — congestion rents cover costs | Private capital (merchant/hybrid) |
| **Track 2** | Not commercially viable, but TSOs are creditworthy | Regulated tariffs / 100% TSO balance sheet |
| **Track 3** | Not commercially viable AND TSO is credit-constrained | CEF grants + EIB loans |

### How to use this notebook

Run each cell in order. Each section explains **what** is being computed and **why**, so you can follow the analytical logic even if you're not familiar with the codebase. All business logic lives in the `grid_financing` Python package — this notebook only orchestrates and displays results.

## Step 0: Setup

We import the `grid_financing` package and configure display settings. The package handles all data loading, calculations, and classification.

In [35]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots

from grid_financing import (
    build_aggregate_summary,
    build_project_master_table,
    build_sensitivity_cases,
    calculate_project_metrics,
    classify_projects,
    export_outputs,
    manual_source_status,
)
from grid_financing.exports import build_triage_scatter_dataset, build_financing_stack_dataset
from grid_financing.classification import TRACK_1, TRACK_2, TRACK_3

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Use notebook_connected renderer (loads plotly.js from CDN, works in VS Code and Jupyter)
pio.renderers.default = "notebook_connected"

## Step 1: Check data sources

Before loading anything, let's verify which core input files are available. The workflow below only shows sources that already exist in this checkout:

- **TYNDP 2024 workbook** — the main project list, investment-level CAPEX, and CBA welfare scenarios
- **TYNDP 2022 workbook** — older CBA scenarios used as fallback
- **Local hourly prices** — country-level day-ahead prices stored in `data/european_wholesale_electricity_price_data_hourly/all_countries.csv`
- **Manual CSV tables** — TSO credit ratings, seeded participant ordering plus CAPEX-share assumptions, and border-zone mappings when available

This notebook uses the local hourly CSV already in the repository. It does **not** call an external API, and it does not need the optional manual day-ahead price table for the current workflow. Missing optional sources are handled later through data-quality flags rather than cluttering this setup view.

In [36]:
source_descriptions = {
    "tyndp2022_workbook": ("TYNDP 2022 workbook", "Older CBA scenarios used as fallback"),
    "tyndp2024_workbook": ("TYNDP 2024 workbook", "Main project list, CAPEX, and CBA scenarios"),
    "local_hourly_prices": ("Local hourly prices", "Country-level hourly price series used for the proxy price spreads"),
    "tso_credit_reference": ("TSO credit reference", "TSO ratings, RAB size, and sovereign indicators"),
    "project_participants": ("Project participants", "Seeded participant ordering and default CAPEX-share assumptions from the TYNDP country field"),
    "project_overrides": ("Project overrides", "Manual overrides for capacities, mappings, or track assignments"),
    "border_zone_map": ("Border-zone map", "Project-to-border mappings used for price processing"),
}

source_status = (
    pd.DataFrame(manual_source_status())
    .query("exists")
    .loc[lambda df: df["source_id"].ne("dayahead_price_inputs")]
    .assign(
        name=lambda df: df["source_id"].map(lambda source_id: source_descriptions[source_id][0]),
        description=lambda df: df["source_id"].map(lambda source_id: source_descriptions[source_id][1]),
        source=lambda df: df["source_id"],
    )
    [["name", "description", "source"]]
    .reset_index(drop=True)
)

source_status

,name,description,source
0,TYNDP 2022 workbook,Older CBA scenarios used as fallback,tyndp2022_workbook
1,TYNDP 2024 workbook,"Main project list, CAPEX, and CBA scenarios",tyndp2024_workbook
2,Local hourly prices,Country-level hourly price series used for the...,local_hourly_prices
3,TSO credit reference,"TSO ratings, RAB size, and sovereign indicators",tso_credit_reference
4,Project participants,Seeded participant ordering and default CAPEX-...,project_participants
5,Border-zone map,Project-to-border mappings used for price proc...,border_zone_map


## Step 2: Build the project master table

This is the core data assembly step. It merges multiple sources into one row per cross-border project in the base case:

1. **Load projects** from the TYNDP 2024 `Trans.Projects` sheet (filters to cross-border only)
2. **Aggregate CAPEX** from `Trans.Investments` (sums investment-level costs per project)
3. **Merge CBA welfare values** from 2024 and 2022 scenario tabs
4. **Apply manual overrides** for missing transfer capacities and country mappings
5. **Attach participant and credit references** — TSO ratings, RAB sizes, sovereign fiscal indicators, and seeded participant CAPEX splits
6. **Preprocess hourly prices** — in development mode, build border spreads from overlapping hourly observations in the local CSV and keep the latest full year for the base case
7. **Flag data-quality issues** — missing CAPEX, missing prices, unresolved multi-country projects, etc.

The year-by-year hourly preprocessing can also expand the table into multiple full-year price scenarios for sensitivity work, for example `proxy_2020` through `proxy_2025`.

**Caveat:** `project_participants.csv` is a seed table, not a verified CBCA register. For two-country projects it mostly applies a default 50/50 split, and for multi-country projects it uses equal-share placeholders unless better project-specific information is entered manually.

In [37]:
master = build_project_master_table(development_mode=True)

print(f"Cross-border projects loaded: {master['project_id'].nunique()}")
print(f"Project rows in base case: {len(master)}")
print(f"Projects with data-quality flags: {master['has_data_quality_issue'].sum()}")
print(f"Price input mode: {master['price_input_mode'].iloc[0]}")
print()

preview = (
    master[[
        "project_id", "project_name", "countries", "status",
        "capex_meur", "capacity_mw",
        "price_scenario", "data_year", "hourly_observation_count",
        "avg_price_diff_eur_per_mwh", "data_quality_flags",
    ]]
    .assign(has_hourly_prices=lambda df: df["hourly_observation_count"].notna())
    .sort_values(["has_hourly_prices", "project_id"], ascending=[False, True])
    .drop(columns="has_hourly_prices")
    .head(10)
)

data_issue_summary = (
    master.loc[master["has_data_quality_issue"], "data_quality_flags"]
    .dropna()
    .str.split(";")
    .explode()
    .value_counts()
    .rename_axis("issue")
    .reset_index(name="project_count")
)

print("Base-case preview (computed hourly-price rows first):")
display(preview)
print("Data issue summary:")
display(data_issue_summary)


Cross-border projects loaded: 100
Project rows in base case: 100
Projects with data-quality flags: 47
Price input mode: development-local-proxy



KeyError: "['hourly_observation_count'] not in index"

### Step 2.1: Explore the pipeline — CAPEX by project status

Before running the triage, let's understand what's in the TYNDP pipeline. The 100 cross-border projects span four development stages:

- **Under Consideration** — earliest stage, no permitting yet
- **Planning** — in planning but not yet in permitting
- **Permitting** — actively going through permitting
- **Construction** — already under construction, likely already financed

**Projects "Under Construction" (~14 bn EUR) are likely already financed**, so the remaining ~156 bn EUR is the relevant financing gap — consistent with the ~150 bn figure cited in the policy brief.

In [ ]:
# CAPEX breakdown by project development status
status_summary = (
    master.groupby("status")
    .agg(
        project_count=("project_id", "nunique"),
        total_capex_meur=("capex_meur", "sum"),
    )
    .sort_values("total_capex_meur", ascending=False)
)
status_summary["total_capex_beur"] = status_summary["total_capex_meur"] / 1000
status_summary["share_pct"] = (
    status_summary["total_capex_meur"] / status_summary["total_capex_meur"].sum() * 100
).round(1)

print(f"Total pipeline: {status_summary['total_capex_meur'].sum()/1000:.1f} bn EUR "
      f"across {status_summary['project_count'].sum()} projects")
print(f"Excluding Construction: "
      f"{(status_summary['total_capex_meur'].sum() - status_summary.loc['Construction', 'total_capex_meur'])/1000:.1f} bn EUR")
print()
status_summary[["project_count", "total_capex_beur", "share_pct"]]

Total pipeline: 169.9 bn EUR across 100 projects
Excluding Construction: 156.1 bn EUR



,project_count,total_capex_beur,share_pct
status,,,
Under Consideration,40,123.214084,72.5
Planning,29,23.561613,13.9
Construction,12,13.790713,8.1
Permitting,19,9.330541,5.5


In [ ]:
# Visualize CAPEX by status
fig_status = px.bar(
    status_summary.reset_index(),
    x="status",
    y="total_capex_beur",
    text="project_count",
    title="CAPEX Pipeline by Project Status (bn EUR)",
    labels={"total_capex_beur": "Total CAPEX (bn EUR)", "status": "Development status"},
    color="status",
    category_orders={"status": ["Under Consideration", "Planning", "Permitting", "Construction"]},
)
fig_status.update_traces(texttemplate="%{text} projects", textposition="outside")
fig_status.update_layout(showlegend=False, height=450)
fig_status.show()

### Step 2.2: CAPEX vs capacity across all projects

A single scatter plot makes it easier to see the full project pipeline at once. The x-axis shows transfer capacity in GW and the y-axis shows project CAPEX in bn EUR.

This highlights whether the pipeline is dominated by a few very large, high-capex links or by many smaller reinforcements.


In [ ]:
scatter_projects = master[[
    "project_id", "project_name", "countries", "country_a", "country_b", "status", "capex_meur", "capacity_mw"
]].copy()
scatter_projects["capacity_gw"] = scatter_projects["capacity_mw"] / 1000
scatter_projects["capex_beur"] = scatter_projects["capex_meur"] / 1000
scatter_projects["corridor_label"] = scatter_projects.apply(
    lambda row: f"{row['country_a']} - {row['country_b']}"
    if pd.notna(row.get("country_a")) and pd.notna(row.get("country_b"))
    else row["countries"],
    axis=1,
)

fig_pipeline = px.scatter(
    scatter_projects,
    x="capacity_gw",
    y="capex_beur",
    color="status",
    hover_name="corridor_label",
    custom_data=["project_name", "status", "capacity_gw", "capex_beur", "project_id"],
    title="Project Pipeline: Capacity vs CAPEX",
    labels={
        "capacity_gw": "Transfer capacity (GW)",
        "capex_beur": "CAPEX (bn EUR)",
        "status": "Development status",
    },
    category_orders={"status": ["Under Consideration", "Planning", "Permitting", "Construction"]},
)
fig_pipeline.update_traces(
    marker={"size": 11, "line": {"width": 0.5, "color": "white"}},
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Project: %{customdata[0]}<br>"
        "Development status: %{customdata[1]}<br>"
        "Transfer capacity: %{customdata[2]:.2f} GW<br>"
        "CAPEX: %{customdata[3]:.2f} bn EUR<br>"
        "Project ID: %{customdata[4]}"
        "<extra></extra>"
    ),
)
fig_pipeline.update_layout(height=520, legend_title_text="Development status")
fig_pipeline.show()


## Step 3: Calculate financing metrics

Now we compute the ratios that drive the triage classification. Here's what each metric means:

| Metric | Formula | Interpretation |
|---|---|---|
| **Capital Recovery Factor** | `r / (1 - (1+r)^(-n))` | Annual repayment as fraction of initial cost (r=5%, n=25yr -> CRF≈0.071) |
| **Annualized CAPEX** | `CAPEX × CRF` | Yearly cost the project must cover to break even |
| **Congestion rate** | `c = 30%` in the base case | Proxy for the share of hours in which the line is congested and can monetize the spread at full transfer capacity |
| **Congestion rent** | `sum_t abs(price_a,t - price_b,t) × capacity × congestion_rate / 1M` | Revenue estimated from the **absolute hourly spread sum**, not from the signed sum |
| **Commercial ratio** | `congestion_rent / annualized_CAPEX` | >1.0 means the project covers its annualized investment cost |
| **Social BCR** | `ΔSEW / annualized_CAPEX` | Welfare benefit per euro of cost (from TYNDP CBA) |
| **Credit constraint score** | `project_CAPEX_share / TSO_RAB` | How large the project is relative to the sponsoring TSO's balance sheet |

The implementation uses `abs(price_a - price_b)` hour by hour and then sums across the year before multiplying by capacity and the congestion-rate assumption.

Here, **congestion rate** is not the observed physical load factor of the line. It is a **proxy for equivalent congested or monetizable hours**: the share of the year in which the interconnector is assumed able to capture the hourly price spread at full transfer capacity.

In the base case, we use a **30% congestion rate**, which is equivalent to about **2,628 hours per year**. This is a simplifying assumption for the congestion-rent screen, and it is deliberately conservative relative to assuming the line monetizes spreads for most of the year.

For the base case, we also annualize CAPEX over **25 years**. That is a pragmatic merchant-finance benchmark: companies generally want the project to cover its investment over roughly **25 years**, not over the full technical lifetime of the asset.

**Caveat:** this congestion-rent estimate is a **majorant**. It is an upper-bound style proxy because it assumes the link can monetize the full absolute hourly spread at the chosen congestion rate, without losses, outages, ramping constraints, directional congestion limits, or other market frictions.


In [ ]:
metrics = calculate_project_metrics(master)

base_congestion_rate = metrics["congestion_rate"].iloc[0]
equivalent_hours = 8760 * base_congestion_rate

print(f"CRF at 5% / 25yr: {metrics['capital_recovery_factor'].iloc[0]:.4f}")
print(f"Base congestion rate: {base_congestion_rate:.0%} (~{equivalent_hours:,.0f} equivalent congested hours/year)")
print(f"Projects with computable commercial ratio: {metrics['commercial_ratio'].notna().sum()} / {len(metrics)}")
print(f"Projects with missing congestion-rent basis: {metrics['congestion_rent_basis'].isna().sum()} / {len(metrics)}")
print()

metrics_preview = (
    metrics[[
        "project_id", "project_name", "price_scenario", "data_year",
        "hourly_observation_count", "congestion_rate", "congestion_rent_basis",
        "annualized_capex_meur_per_year",
        "estimated_congestion_rent_meur_per_year",
        "commercial_ratio", "social_bcr",
        "credit_constraint_score", "credit_constrained",
    ]]
    .assign(has_congestion_metric=lambda df: df["congestion_rent_basis"].notna())
    .sort_values(["has_congestion_metric", "project_id"], ascending=[False, True])
    .drop(columns="has_congestion_metric")
)

metrics_preview.head(15)


CRF at 5% / 25yr: 0.0710
Base congestion rate: 30% (~2,628 equivalent congested hours/year)
Projects with computable commercial ratio: 76 / 100
Projects with missing congestion-rent basis: 20 / 100



,project_id,project_name,price_scenario,data_year,hourly_observation_count,congestion_rate,congestion_rent_basis,annualized_capex_meur_per_year,estimated_congestion_rent_meur_per_year,commercial_ratio,social_bcr,credit_constraint_score,credit_constrained
0,4.0,Interconnection Portugal-Spain,proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,7.888494,5.311732,0.673352,3.295940,0.015883,False
1,16.0,Biscay Gulf,proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,219.952618,117.516485,0.534281,1.900409,0.166667,True
2,28.0,Italy-Montenegro,proxy_2024,2024.0,8784.0,0.3,hourly_price_sum,30.083842,31.127008,1.034675,0.199443,0.009422,False
4,40.0,Belgium-Luxembourg-Germany: long-term perspective,proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,14.900016,16.392897,1.100193,1.476508,NaN,False
5,47.0,Westtirol (AT) - Vöhringen (DE),proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,0.730810,20.564327,28.139076,47.892045,0.001717,True
6,81.0,North South Interconnector,proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,25.755742,79.792789,3.098058,0.155305,0.051857,True
7,94.0,GerPol Improvements,proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,7.861532,78.480531,9.982854,17.808233,0.011080,True
8,107.0,Celtic Interconnector,proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,115.155838,106.848027,0.927856,2.110184,0.231857,True
9,111.0,Aurora line (3rd AC Finland-Sweden north),proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,19.157163,50.752656,2.649278,2.401191,0.038571,True
10,121.0,Nautilus: multi-purpose interconnector Belgium...,proxy_2025,2025.0,8760.0,0.3,hourly_price_sum,70.952457,73.973714,1.042581,1.409394,0.027027,False


### Step 3.1: Congestion rent heatmap by country pair (2015-2021)

To see how congestion-rent potential moves over time, we rebuild the yearly price scenarios from 2015 to 2021 and aggregate the estimated congestion rent by country pair.

The heatmaps below use only project-year rows with a full overlapping hourly year. Country pairs are mirrored so each bilateral corridor appears symmetrically in the matrix. Values are in MEUR per year.


In [ ]:
heatmap_years = range(2015, 2022)
yearly_master = build_project_master_table(development_mode=True, price_years=heatmap_years)
yearly_master = yearly_master[
    yearly_master["hourly_observation_count"].notna()
    & yearly_master["country_a"].notna()
    & yearly_master["country_b"].notna()
].copy()
yearly_metrics = calculate_project_metrics(yearly_master)

country_pairs = yearly_metrics.apply(
    lambda row: tuple(sorted((str(row["country_a"]), str(row["country_b"])))),
    axis=1,
)
yearly_metrics[["country_row", "country_col"]] = pd.DataFrame(country_pairs.tolist(), index=yearly_metrics.index)

pair_rents = (
    yearly_metrics.groupby(["data_year", "country_row", "country_col"], dropna=False)["estimated_congestion_rent_meur_per_year"]
    .sum()
    .reset_index()
)

mirrored_pair_rents = pd.concat([
    pair_rents,
    pair_rents.rename(columns={"country_row": "country_col", "country_col": "country_row"}),
], ignore_index=True).drop_duplicates(["data_year", "country_row", "country_col"])

countries = sorted(set(mirrored_pair_rents["country_row"]).union(mirrored_pair_rents["country_col"]))
zmax = mirrored_pair_rents["estimated_congestion_rent_meur_per_year"].max()

fig_heatmap = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=[str(year) for year in heatmap_years],
    horizontal_spacing=0.04,
    vertical_spacing=0.08,
)

for index, year in enumerate(heatmap_years):
    row = index // 3 + 1
    col = index % 3 + 1
    matrix = (
        mirrored_pair_rents.loc[mirrored_pair_rents["data_year"] == year]
        .pivot(index="country_row", columns="country_col", values="estimated_congestion_rent_meur_per_year")
        .reindex(index=countries, columns=countries, fill_value=0)
    )
    fig_heatmap.add_trace(
        go.Heatmap(
            z=matrix.values,
            x=matrix.columns,
            y=matrix.index,
            coloraxis="coloraxis",
            zmin=0,
            zmax=zmax,
            hovertemplate="Row country: %{y}<br>Column country: %{x}<br>Congestion rent: %{z:.1f} MEUR/year<extra></extra>",
        ),
        row=row,
        col=col,
    )

fig_heatmap.update_layout(
    height=1100,
    title="Congestion rent by country pair and year (MEUR/year)",
    coloraxis={"colorscale": "YlOrRd", "colorbar": {"title": "MEUR/year"}},
)
fig_heatmap.show()


### Step 3.2: What is TSO RAB and how does the credit-constraint metric work?

**TSO RAB** means the **regulated asset base** of the transmission system operator. In practice, it is a proxy for the size of the regulated balance sheet on which the TSO can earn an allowed return.

The credit-constraint score asks a simple question for each project side: **how large is that side of the project compared with the sponsoring TSO's RAB?**

`credit constraint score = project CAPEX share / TSO RAB`

A higher score means the project is large relative to the TSO's balance sheet. In the base case, a score above **0.15** means the project side is more than **15%** of that TSO's RAB, which is our main quantitative flag for a possible financing constraint.

For transparency, the notebook uses the **country-side mapping attached to each project row**: side **A** uses the first participant in `project_participants.csv`, while side **B** uses the second participant. Their associated countries are then matched to the TSO credit reference table to populate `tso_a_name_reference` and `tso_b_name_reference`.

At project level, the notebook keeps the **binding side's score**, i.e. `max(score_a, score_b)`. That is deliberate: the project is treated as credit-constrained if **either** side needs support, so the binding number is the more constrained side, not the less constrained one. Using the minimum would describe the stronger side and would understate the financing bottleneck.

**Important caveat:** this participant table is a **seed methodology**, not a verified sponsor-allocation dataset. It is built from the TYNDP `Country` field with default CAPEX-share assumptions: typically **50/50 for two-country projects** and **equal shares for multi-country projects** unless better manual information is available. For projects with three or more participants, the current pipeline still reduces the financing screen to the first two ordered participants, so the credit-constraint result is a simplification.

**How this should be better covered:** the next improvement is to replace those placeholder shares with actual CBCA or sponsor allocations and move from the current `A/B` abstraction to a true participant-level financing allocation for all project participants.

In [ ]:
credit_sides = pd.concat([
    metrics[[
        "project_id", "project_name", "countries", "status",
        "country_a", "country_a_participant",
        "project_capex_share_a_meur", "tso_a_rab_beur", "credit_constraint_score_a",
        "tso_a_name_reference",
    ]].rename(columns={
        "country_a": "border_country",
        "country_a_participant": "participant_country",
        "project_capex_share_a_meur": "project_capex_share_meur",
        "tso_a_rab_beur": "tso_rab_beur",
        "credit_constraint_score_a": "credit_constraint_score_side",
        "tso_a_name_reference": "tso_name",
    }).assign(side="A"),
    metrics[[
        "project_id", "project_name", "countries", "status",
        "country_b", "country_b_participant",
        "project_capex_share_b_meur", "tso_b_rab_beur", "credit_constraint_score_b",
        "tso_b_name_reference",
    ]].rename(columns={
        "country_b": "border_country",
        "country_b_participant": "participant_country",
        "project_capex_share_b_meur": "project_capex_share_meur",
        "tso_b_rab_beur": "tso_rab_beur",
        "credit_constraint_score_b": "credit_constraint_score_side",
        "tso_b_name_reference": "tso_name",
    }).assign(side="B"),
], ignore_index=True)

credit_sides["country_side"] = credit_sides["participant_country"].fillna(credit_sides["border_country"])
credit_sides = credit_sides[
    credit_sides["project_capex_share_meur"].notna() & credit_sides["tso_rab_beur"].notna()
].copy()
credit_sides["threshold_exceeded"] = credit_sides["credit_constraint_score_side"] > 0.15
credit_sides["corridor_label"] = credit_sides["countries"].fillna(credit_sides["project_name"])
credit_sides["threshold_label"] = credit_sides["threshold_exceeded"].map({
    False: "Within 15% threshold",
    True: "Above 15% threshold",
})

credit_side_mapping = credit_sides[[
    "project_name", "countries", "side", "country_side", "tso_name",
    "tso_rab_beur", "project_capex_share_meur", "credit_constraint_score_side",
]].rename(columns={"credit_constraint_score_side": "credit_constraint_score"}).head(12)
credit_side_mapping

fig_credit = px.scatter(
    credit_sides,
    x="tso_rab_beur",
    y="project_capex_share_meur",
    color="threshold_label",
    hover_name="corridor_label",
    custom_data=[
        "project_name",
        "side",
        "country_side",
        "tso_name",
        "tso_rab_beur",
        "project_capex_share_meur",
        "credit_constraint_score_side",
    ],
    title="Project CAPEX Share vs TSO RAB by Project Side",
    labels={
        "tso_rab_beur": "TSO RAB (bn EUR)",
        "project_capex_share_meur": "Project CAPEX share (MEUR)",
        "threshold_label": "Threshold status",
    },
    color_discrete_map={"Within 15% threshold": "#4c78a8", "Above 15% threshold": "#d62728"},
)
xmax = credit_sides["tso_rab_beur"].max()
fig_credit.add_scatter(
    x=[0, xmax],
    y=[0, 150 * xmax],
    mode="lines",
    name="15% threshold",
    line={"dash": "dash", "color": "gray"},
    hoverinfo="skip",
)
fig_credit.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Project: %{customdata[0]}<br>"
        "Project side: %{customdata[1]}<br>"
        "Country side: %{customdata[2]}<br>"
        "TSO: %{customdata[3]}<br>"
        "TSO RAB: %{customdata[4]:.1f} bn EUR<br>"
        "Project CAPEX share: %{customdata[5]:.1f} MEUR<br>"
        "Credit-constraint score: %{customdata[6]:.3f}"
        "<extra></extra>"
    ),
    selector={"mode": "markers"},
)
fig_credit.update_layout(height=520, legend_title_text="Threshold status")
fig_credit.show()

most_constrained_sides = credit_sides.nlargest(12, "credit_constraint_score_side")[[
    "project_name", "countries", "side", "country_side", "tso_name",
    "tso_rab_beur", "project_capex_share_meur", "credit_constraint_score_side",
]].rename(columns={"credit_constraint_score_side": "credit_constraint_score"})
most_constrained_sides


,project_name,countries,side,country_side,tso_name,tso_rab_beur,project_capex_share_meur,credit_constraint_score
95,NaN,MT ; TN,A,MT,Enemalta plc,0.2,744.50,3.722500
92,NaN,HU ; RO,A,HU,MAVIR ZRt,1.5,3642.00,2.428000
192,NaN,HU ; RO,B,RO,Transelectrica SA,1.5,3642.00,2.428000
196,NaN,DE ; GR,B,GR,ADMIE/IPTO (Independent Power Transmission Ope...,2.0,4050.00,2.025000
65,Off-shore Wind Park in Latvia and Estonia - EL...,EE ; LV,A,EE,Elering AS,0.5,612.50,1.225000
67,Estlink 3,EE ; FI,A,EE,Elering AS,0.5,533.00,1.066000
166,Offshore Hybrid HVDC Interconnector BEDK,BE ; DK,B,DK,Energinet,5.0,5046.00,1.009200
155,GREGY Green Energy Interconnector | Purifying ...,EG ; GR,B,GR,ADMIE/IPTO (Independent Power Transmission Ope...,2.0,1784.50,0.892250
165,Off-shore Wind Park in Latvia and Estonia - EL...,EE ; LV,B,LV,AST (AS Augstsprieguma tikls),0.8,612.50,0.765625
164,Malta-Italy Cable Link No.2,IT ; MT,B,MT,Enemalta plc,0.2,142.75,0.713750


## Step 4: Classify projects into financing tracks

The classification logic is straightforward:

1. **Commercial ratio > 1.0?** → **Track 1** (market-financed). Congestion rents are high enough to cover the annualized investment cost under a merchant or hybrid structure.
2. **Not commercially viable, but TSO is creditworthy?** → **Track 2** (regulated + CBCA). The TSO can finance through regulated tariffs with a Cross-Border Cost Allocation agreement.
3. **Not commercially viable AND credit-constrained?** → **Track 3** (needs EU support). The TSO's balance sheet is too small relative to project cost, or the sovereign is fiscally stressed. These need CEF grants + EIB loans.
4. **Missing data?** → **Unclassified**. Projects without price data or transfer capacity can't be evaluated.

A project is "credit-constrained" if **any** of these hold for **either side**:
- Credit constraint score > 0.15 (project is >15% of TSO RAB)
- TSO has sub-investment-grade rating
- Sovereign has debt >100% GDP AND deficit >3% GDP

After classification, the financing stack is estimated per project. **Track 2 defaults to 100% TSO balance-sheet financing** under a regulated tariff framework, while **Track 3** CEF grant rates default to 85% for cohesion-country sides and 50% otherwise.

In [ ]:
classified = classify_projects(metrics)

print("Track distribution:")
print(classified["financing_track"].value_counts().to_string())
print()

classified[[
    "project_id", "project_name", "countries",
    "capex_meur", "commercial_ratio", "social_bcr",
    "credit_constraint_score", "financing_track",
    "estimated_cef_grant_meur", "data_quality_flags",
]].head(15)

Track distribution:
financing_track
Track 1: Market-financed (merchant/hybrid)           47
Unclassified: insufficient data                      24
Track 3: Credit-constrained - targeted EU support    15
Track 2: Regulated + CBCA                            14



,project_id,project_name,countries,capex_meur,commercial_ratio,social_bcr,credit_constraint_score,financing_track,estimated_cef_grant_meur,data_quality_flags
0,4.0,Interconnection Portugal-Spain,ES ; PT,111.18,0.673352,3.295940,0.015883,Track 2: Regulated + CBCA,0.00,<NA>
1,16.0,Biscay Gulf,ES ; FR,3100.00,0.534281,1.900409,0.166667,Track 3: Credit-constrained - targeted EU support,1550.00,<NA>
2,28.0,Italy-Montenegro,IT ; ME,424.00,1.034675,0.199443,0.009422,Track 1: Market-financed (merchant/hybrid),0.00,missing_credit_reference
3,29.0,Italy-Tunisia,IT ; TN,850.00,NaN,0.829056,0.018889,Unclassified: insufficient data,0.00,missing_credit_reference;missing_price_inputs
4,40.0,Belgium-Luxembourg-Germany: long-term perspective,BE ; DE ; LU,210.00,1.100193,1.476508,NaN,Track 1: Market-financed (merchant/hybrid),0.00,missing_credit_reference
5,47.0,Westtirol (AT) - Vöhringen (DE),AT ; DE,10.30,28.139076,47.892045,0.001717,Track 1: Market-financed (merchant/hybrid),0.00,<NA>
6,81.0,North South Interconnector,GB ; IE,363.00,3.098058,0.155305,0.051857,Track 1: Market-financed (merchant/hybrid),0.00,missing_credit_reference
7,94.0,GerPol Improvements,DE ; PL,110.80,9.982854,17.808233,0.011080,Track 1: Market-financed (merchant/hybrid),0.00,<NA>
8,107.0,Celtic Interconnector,FR ; IE,1623.00,0.927856,2.110184,0.231857,Track 3: Credit-constrained - targeted EU support,811.50,<NA>
9,111.0,Aurora line (3rd AC Finland-Sweden north),FI ; SE,270.00,2.649278,2.401191,0.038571,Track 1: Market-financed (merchant/hybrid),0.00,<NA>


## Step 5: Visualize the triage results

### 5.1 Scatter plot — the core policy chart

This is the centrepiece visualization for the policy brief. Each dot is a project:

- **X-axis:** commercial viability ratio. Right of the dashed line at 1.0 = congestion rents cover annualized project cost.
- **Y-axis:** binding credit-constraint score. Above the dashed line at 0.15 = at least one project side is materially large relative to its sponsoring TSO's RAB.
- **Dot size:** total CAPEX.
- **Color:** financing track.

The display caps are controlled in the parameter cell just below, so you can adjust them yourself. The chart only clips the plotted view; hover still shows the original underlying values for clipped points.

The quadrants map directly to tracks:
- **Right half** → Track 1 (commercially viable, any credit profile)
- **Bottom-left** → Track 2 (not viable, but creditworthy TSO)
- **Top-left** → Track 3 (not viable AND credit-constrained)


In [ ]:
COMMERCIAL_RATIO_DISPLAY_CAP = 3
CREDIT_CONSTRAINT_DISPLAY_CAP = 2

print(f"Commercial-ratio display cap: {COMMERCIAL_RATIO_DISPLAY_CAP}")
print(f"Credit-score display cap: {CREDIT_CONSTRAINT_DISPLAY_CAP}")


Commercial-ratio display cap: 3
Credit-score display cap: 2


In [ ]:
scatter_df = build_triage_scatter_dataset(classified)
scatter_plot = (
    scatter_df[
        scatter_df["commercial_ratio"].notna() & scatter_df["credit_constraint_score"].notna()
    ]
    .copy()
    .assign(
        financing_track_display=lambda df: df["financing_track"].map({
            TRACK_1: "Track 1: Market-financed",
            TRACK_2: "Track 2: TSO-financed",
            TRACK_3: "Track 3: Targeted EU support",
        }),
        commercial_ratio_display=lambda df: df["commercial_ratio"].clip(upper=COMMERCIAL_RATIO_DISPLAY_CAP),
        credit_constraint_score_display=lambda df: df["credit_constraint_score"].clip(upper=CREDIT_CONSTRAINT_DISPLAY_CAP),
        commercial_ratio_clipped=lambda df: df["commercial_ratio"] > COMMERCIAL_RATIO_DISPLAY_CAP,
        credit_constraint_score_clipped=lambda df: df["credit_constraint_score"] > CREDIT_CONSTRAINT_DISPLAY_CAP,
    )
)

fig_scatter = px.scatter(
    scatter_plot,
    x="commercial_ratio_display",
    y="credit_constraint_score_display",
    size="capex_meur",
    color="financing_track_display",
    hover_name="corridor_label",
    custom_data=[
        "project_name",
        "commercial_ratio",
        "credit_constraint_score",
        "social_bcr",
        "capex_meur",
    ],
    title="Financing Triage: Commercial Viability vs Credit Constraint",
    labels={
        "commercial_ratio_display": "Commercial viability ratio",
        "credit_constraint_score_display": "Binding credit-constraint score",
        "financing_track_display": "Financing track",
    },
    category_orders={
        "financing_track_display": [
            "Track 1: Market-financed",
            "Track 2: TSO-financed",
            "Track 3: Targeted EU support",
        ]
    },
    color_discrete_map={
        "Track 1: Market-financed": "#1f77b4",
        "Track 2: TSO-financed": "#ff7f0e",
        "Track 3: Targeted EU support": "#d62728",
    },
)
fig_scatter.add_vline(x=1.0, line_dash="dash", line_color="gray",
                       annotation_text="Commercial threshold (1.0)")
fig_scatter.add_hline(y=0.15, line_dash="dash", line_color="gray",
                       annotation_text="Credit constraint threshold (0.15)")
fig_scatter.update_traces(
    marker={"line": {"width": 0.5, "color": "white"}},
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Project: %{customdata[0]}<br>"
        "Financing track: %{fullData.name}<br>"
        "Commercial viability ratio: %{customdata[1]:.2f}<br>"
        "Binding credit-constraint score: %{customdata[2]:.3f}<br>"
        "Social BCR: %{customdata[3]:.2f}<br>"
        "CAPEX: %{customdata[4]:,.0f} MEUR"
        "<extra></extra>"
    ),
    selector={"mode": "markers"},
)
fig_scatter.update_xaxes(
    range=[0, COMMERCIAL_RATIO_DISPLAY_CAP],
    title_text="Commercial viability ratio<br><sup>Congestion rent / annualized CAPEX</sup>",
)
fig_scatter.update_yaxes(
    range=[0, CREDIT_CONSTRAINT_DISPLAY_CAP],
    title_text="Binding credit-constraint score<br><sup>Project CAPEX share / TSO RAB</sup>",
)
fig_scatter.update_layout(
    height=620,
    legend_title_text="Financing track",
    annotations=list(fig_scatter.layout.annotations) + [
        dict(
            text=(
                f"Display caps for readability: x<={COMMERCIAL_RATIO_DISPLAY_CAP} "
                f"({int(scatter_plot['commercial_ratio_clipped'].sum())} clipped), "
                f"y<={CREDIT_CONSTRAINT_DISPLAY_CAP} "
                f"({int(scatter_plot['credit_constraint_score_clipped'].sum())} clipped)"
            ),
            x=1,
            xref="paper",
            y=-0.18,
            yref="paper",
            xanchor="right",
            showarrow=False,
            font={"size": 11, "color": "gray"},
        )
    ],
)
fig_scatter.show()


### 5.2 Aggregate financing needs

How does the ~170 bn EUR pipeline break down by financing track and source?

The key policy insight: **if merchant financing works for Track 1 projects, CEF resources can be concentrated on the credit-constrained projects (Track 3) that need them most.**

In [ ]:
summary = build_aggregate_summary(classified)
summary[[
    "financing_track", "project_count",
    "total_capex_beur", "total_cef_beur", "total_eib_beur",
    "total_private_beur", "total_tso_beur",
]]

,financing_track,project_count,total_capex_beur,total_cef_beur,total_eib_beur,total_private_beur,total_tso_beur
0,Track 1: Market-financed (merchant/hybrid),47,24.380745,0.000000,0.000000,24.380745,0.000000
1,Track 2: Regulated + CBCA,14,31.377874,0.000000,0.000000,0.000000,31.377874
2,Track 3: Credit-constrained - targeted EU support,15,40.402190,26.415485,8.080438,0.000000,6.794037
3,Unclassified: insufficient data,24,73.736142,0.000000,0.000000,0.000000,0.000000
4,Total,100,169.896951,26.415485,8.080438,24.380745,38.171911


### 5.3 Financing stack chart

Stacked bar showing the financing breakdown per track. Under the current assumptions, **Track 2 is entirely TSO-financed**, so public support should only appear in **Track 3**. Compare the Track 3 CEF grant total with the proposed CEF-E budget (~17 bn EUR).

In [ ]:
stack_df = build_financing_stack_dataset(summary)

fig_stack = px.bar(
    stack_df,
    x="financing_track_label",
    y="value_beur",
    color="financing_component",
    barmode="stack",
    custom_data=["financing_track_display"],
    title="Aggregate Financing Stack by Track",
    labels={
        "value_beur": "Amount (bn EUR)",
        "financing_track_label": "Financing track",
        "financing_component": "Funding source",
    },
    category_orders={
        "financing_track_label": [
            "Track 1<br>Market-financed",
            "Track 2<br>TSO-financed",
            "Track 3<br>Targeted EU support",
        ]
    },
)
fig_stack.update_traces(
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Funding source: %{fullData.name}<br>"
        "Amount: %{y:.2f} bn EUR"
        "<extra></extra>"
    )
)
fig_stack.update_layout(height=500, legend_title_text="Funding source")
fig_stack.update_xaxes(title_text="Financing track")
fig_stack.update_yaxes(title_text="Amount (bn EUR)")
fig_stack.show()


## Step 6: Sensitivity analysis

The thresholds and assumptions above are inherently uncertain. Here we combine financing assumptions with full-year hourly price windows from 2020 to 2025:

- **Price years:** 2020, 2021, 2022, 2023, 2024, 2025 — changes congestion rents using observed full-year hourly spreads from the local dataset
- **Discount rates:** 4%, 5% (base), 6% — changes annualized CAPEX and therefore all ratios
- **Congestion-rate assumptions:** 30% (base), 50%, 70% — changes the assumed share of hours in which the line can monetize spreads
- **Credit thresholds:** 10%, 15% (base), 20% — shifts projects between Track 2 and 3

Only project-year rows with a full overlapping hourly year are kept in this sensitivity table. That yields up to `6 × 3 × 3 × 3 = 162` combinations per project where full-year price data exist.


In [ ]:
price_years = range(2020, 2026)
sensitivity_master = build_project_master_table(development_mode=True, price_years=price_years)
sensitivity_master = sensitivity_master[sensitivity_master["hourly_observation_count"].notna()].copy()
sensitivity = build_sensitivity_cases(sensitivity_master, price_scenarios=tuple(f"proxy_{year}" for year in price_years))

print(f"Sensitivity source rows with full-year hourly data: {len(sensitivity_master)}")
print(f"Sensitivity cases: {sensitivity['sensitivity_case'].nunique()}")
print(f"Sensitivity output rows: {len(sensitivity)}")
print()

sensitivity[[
    "project_id", "project_name", "price_scenario", "data_year",
    "hourly_observation_count", "commercial_ratio",
    "credit_constraint_score", "credit_constrained",
]].head(15)


Sensitivity source rows with full-year hourly data: 456
Sensitivity cases: 162
Sensitivity output rows: 12312



,project_id,project_name,price_scenario,data_year,hourly_observation_count,commercial_ratio,credit_constraint_score,credit_constrained
0,4.0,Interconnection Portugal-Spain,proxy_2020,2020,8784.0,0.066862,0.015883,False
1,16.0,Biscay Gulf,proxy_2020,2020,8784.0,0.170944,0.166667,True
2,40.0,Belgium-Luxembourg-Germany: long-term perspective,proxy_2020,2020,8784.0,0.414093,NaN,False
3,47.0,Westtirol (AT) - Vöhringen (DE),proxy_2020,2020,8784.0,7.88512,0.001717,True
4,81.0,North South Interconnector,proxy_2020,2020,8784.0,1.15793,0.051857,True
5,94.0,GerPol Improvements,proxy_2020,2020,8784.0,9.289703,0.011080,True
6,107.0,Celtic Interconnector,proxy_2020,2020,8784.0,0.234142,0.231857,True
7,111.0,Aurora line (3rd AC Finland-Sweden north),proxy_2020,2020,8784.0,1.062656,0.038571,True
8,121.0,Nautilus: multi-purpose interconnector Belgium...,proxy_2020,2020,8784.0,0.48455,0.027027,False
9,124.0,NordBalt phase 2,proxy_2020,2020,8784.0,0.0,NaN,False


## Step 7: Export results

Write all outputs to disk for the policy brief:

- `data/processed/project_master_table.csv` — full project-level table with all metrics and flags
- `data/processed/aggregate_summary.csv` — per-track totals
- `outputs/tables/` — Excel versions for reviewers
- `outputs/charts/` — interactive HTML scatter and stacked-bar charts

All exported tables include assumptions, source provenance, and data-quality flags so reviewers can trace any metric back to its source.

In [ ]:
exports = export_outputs(classified)

print("Exported files:")
for name, path in exports.items():
    print(f"  {name}: {path}")

Exported files:
  project_master_table_csv: /Users/lucas/Workspace/PolicyBrief/GridFinancing/data/processed/project_master_table.csv
  aggregate_summary_csv: /Users/lucas/Workspace/PolicyBrief/GridFinancing/data/processed/aggregate_summary.csv
  project_level_results_xlsx: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/tables/project_level_results.xlsx
  aggregate_summary_xlsx: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/tables/aggregate_summary.xlsx
  triage_scatter_html: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/charts/triage_scatter.html
  aggregate_financing_stack_html: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/charts/aggregate_financing_stack.html
